In [1]:
import pyspark
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder.config('spark.driver.host','localhost').appName('10').getOrCreate()

In [3]:
videos =  spark.read.csv('C:\\datas\\3rd_week_data\\youtubevideos.csv',inferSchema=True,header=True)

In [4]:
videos.printSchema()

root
 |-- video_id: string (nullable = true)
 |-- trending_date: string (nullable = true)
 |-- title: string (nullable = true)
 |-- channel_title: string (nullable = true)
 |-- category_id: string (nullable = true)
 |-- publish_time: string (nullable = true)
 |-- tags: string (nullable = true)
 |-- views: string (nullable = true)
 |-- likes: string (nullable = true)
 |-- dislikes: string (nullable = true)
 |-- comment_count: string (nullable = true)
 |-- thumbnail_link: string (nullable = true)
 |-- comments_disabled: string (nullable = true)
 |-- ratings_disabled: string (nullable = true)
 |-- video_error_or_removed: string (nullable = true)
 |-- description: string (nullable = true)



In [5]:
from pyspark.sql.functions import *

In [6]:
print(videos.select('publish_time').collect()[0][0])

2017-11-13T17:13:01.000Z


In [7]:
videos = videos.withColumn('publish_time_2',regexp_replace(videos.publish_time,'T',' '))
videos = videos.withColumn('publish_time_2',regexp_replace(videos.publish_time_2,'Z',''))
#videos = videos.withColumn('publish_time_2',regexp_replace(videos.publish_time,'T',' ')).withColumn('publish_time_2',regexp_replace('publish_time_2','Z',''))
print(videos.select('publish_time_2').collect()[0][0])

2017-11-13 17:13:01.000


In [8]:
videos = videos.withColumn('publish_time_3',to_timestamp(videos.publish_time_2,'yyyy-MM-dd HH:mm:ss.SSS'))
# to_timestamp: yyyy-MM-dd HH:mm:SS
videos.select('publish_time','publish_time_2','publish_time_3').show(5)
videos.printSchema()

+--------------------+--------------------+-------------------+
|        publish_time|      publish_time_2|     publish_time_3|
+--------------------+--------------------+-------------------+
|2017-11-13T17:13:...|2017-11-13 17:13:...|2017-11-13 17:13:01|
|2017-11-13T07:30:...|2017-11-13 07:30:...|2017-11-13 07:30:00|
|2017-11-12T19:05:...|2017-11-12 19:05:...|2017-11-12 19:05:24|
|2017-11-13T11:00:...|2017-11-13 11:00:...|2017-11-13 11:00:04|
|2017-11-12T18:01:...|2017-11-12 18:01:...|2017-11-12 18:01:41|
+--------------------+--------------------+-------------------+
only showing top 5 rows

root
 |-- video_id: string (nullable = true)
 |-- trending_date: string (nullable = true)
 |-- title: string (nullable = true)
 |-- channel_title: string (nullable = true)
 |-- category_id: string (nullable = true)
 |-- publish_time: string (nullable = true)
 |-- tags: string (nullable = true)
 |-- views: string (nullable = true)
 |-- likes: string (nullable = true)
 |-- dislikes: string (nullabl

In [9]:
videos = videos.withColumn('publish_time_4',translate(videos.publish_time,'TZ',' '))
videos.select('publish_time_4').collect()[0][0]
# translate(column,'abc','def'): a->d, b->e, c->f
# translate(videos.publish_time,'TZ',' '): T -> ' ', Z -> 삭제

'2017-11-13 17:13:01.000'

In [10]:
from pyspark.sql.types import *

In [11]:
videos = videos.withColumn('likes',videos.likes.cast(IntegerType()))\
.withColumn('dislikes',videos.dislikes.cast(IntegerType()))

In [12]:
videos.printSchema()

root
 |-- video_id: string (nullable = true)
 |-- trending_date: string (nullable = true)
 |-- title: string (nullable = true)
 |-- channel_title: string (nullable = true)
 |-- category_id: string (nullable = true)
 |-- publish_time: string (nullable = true)
 |-- tags: string (nullable = true)
 |-- views: string (nullable = true)
 |-- likes: integer (nullable = true)
 |-- dislikes: integer (nullable = true)
 |-- comment_count: string (nullable = true)
 |-- thumbnail_link: string (nullable = true)
 |-- comments_disabled: string (nullable = true)
 |-- ratings_disabled: string (nullable = true)
 |-- video_error_or_removed: string (nullable = true)
 |-- description: string (nullable = true)
 |-- publish_time_2: string (nullable = true)
 |-- publish_time_3: timestamp (nullable = true)
 |-- publish_time_4: string (nullable = true)



In [13]:
videos = videos.withColumn('favorability',when(videos.likes > videos.dislikes,'good').otherwise('bad'))
# when 계속 붙여서 조건식 만드는 것도 가능
videos.select('likes','dislikes','favorability').show(10)

+------+--------+------------+
| likes|dislikes|favorability|
+------+--------+------------+
| 57527|    2966|        good|
| 97185|    6146|        good|
|146033|    5339|        good|
| 10172|     666|        good|
|132235|    1989|        good|
|  9763|     511|        good|
| 15993|    2445|        good|
| 23663|     778|        good|
|  3543|     119|        good|
| 12654|    1363|        good|
+------+--------+------------+
only showing top 10 rows



In [14]:
videos = videos.select('*',expr('CASE WHEN likes > dislikes THEN "good" ELSE "bad" END AS favorability_2'))
videos.printSchema()
# withColumn은 새로운 열 생성해 기존 데이터 프레임에 붙임, select expr은 select한 열과 조건식으로 만든 열을 포함한 새로운 데이터 프레임을 반환함
# selectExpr('likes','dislikes',('CASE WHEN likes > dislikes THEN "good" ELSE "bad" END AS favorability_2'))

root
 |-- video_id: string (nullable = true)
 |-- trending_date: string (nullable = true)
 |-- title: string (nullable = true)
 |-- channel_title: string (nullable = true)
 |-- category_id: string (nullable = true)
 |-- publish_time: string (nullable = true)
 |-- tags: string (nullable = true)
 |-- views: string (nullable = true)
 |-- likes: integer (nullable = true)
 |-- dislikes: integer (nullable = true)
 |-- comment_count: string (nullable = true)
 |-- thumbnail_link: string (nullable = true)
 |-- comments_disabled: string (nullable = true)
 |-- ratings_disabled: string (nullable = true)
 |-- video_error_or_removed: string (nullable = true)
 |-- description: string (nullable = true)
 |-- publish_time_2: string (nullable = true)
 |-- publish_time_3: timestamp (nullable = true)
 |-- publish_time_4: string (nullable = true)
 |-- favorability: string (nullable = false)
 |-- favorability_2: string (nullable = false)



In [15]:
videos = videos.select('*',to_date('trending_date','yy.dd.MM').alias('trending_date_2'))
# to_date: yyyy-MM-dd
videos.printSchema()
videos.select('trending_date','trending_date_2').show(10)

root
 |-- video_id: string (nullable = true)
 |-- trending_date: string (nullable = true)
 |-- title: string (nullable = true)
 |-- channel_title: string (nullable = true)
 |-- category_id: string (nullable = true)
 |-- publish_time: string (nullable = true)
 |-- tags: string (nullable = true)
 |-- views: string (nullable = true)
 |-- likes: integer (nullable = true)
 |-- dislikes: integer (nullable = true)
 |-- comment_count: string (nullable = true)
 |-- thumbnail_link: string (nullable = true)
 |-- comments_disabled: string (nullable = true)
 |-- ratings_disabled: string (nullable = true)
 |-- video_error_or_removed: string (nullable = true)
 |-- description: string (nullable = true)
 |-- publish_time_2: string (nullable = true)
 |-- publish_time_3: timestamp (nullable = true)
 |-- publish_time_4: string (nullable = true)
 |-- favorability: string (nullable = false)
 |-- favorability_2: string (nullable = false)
 |-- trending_date_2: date (nullable = true)

+-------------+----------